In [1]:
include("../RayTracing.jl")

Main.RayTracing

In [16]:
# Blindly following PBRT because apparently I am having trouble thinking for myself
struct Distribution1D_OLD
    func::Vector{Float64}
    cdf::Vector{Float64}
    func_int::Float64

    function Distribution1D_OLD(func::Vector{Float64})
        N = length(func)
        cdf = Array{Float64, 1}(undef, N+1)
        cdf[1] = 0
        for i = 2:(N+1)
            cdf[i] = cdf[i-1] + func[i-1] / N
        end
        # print("$(func)\n\n")
        func_int = cdf[N+1]
        if func_int == 0
            for i = 1:(N+1)
                cdf[i] = i / (N+1)
            end
        else
            for i = 1:(N+1)
                cdf[i] /= func_int
            end
        end
        return new(func,cdf,func_int)
    end
end

function sample_continuous(d::Distribution1D_OLD, u::Float64)::Tuple{Float64, Float64, Int64} # (val, pdf, offset)
    offset = min(sum(d.cdf .<= u),length(d.cdf)-1) # John hack to avoid index error when u=1.0
    du = u - d.cdf[offset]
    if d.cdf[offset+1] - d.cdf[offset] > 0
        du /= (d.cdf[offset+1] - d.cdf[offset])
    end
    pdf_val = (d.func_int > 0) ? d.func[offset] / d.func_int : 0 
    return (offset-1 + du) / length(d.func), pdf_val, offset
end

function sample_discrete(d::Distribution1D_OLD, u::Float64)::Tuple{Int64, Float64, Float64}
    offset = min(sum(d.cdf .<= u),length(d.cdf)-1) # John hack to avoid index error when u=1.0
    pdf_val = (d.func_int > 0) ? d.func[offset] / (d.func_int * length(d.func)) : 0
    val = (u - d.cdf[offset]) / (d.cdf[offset + 1] - d.cdf[offset])
    return offset, pdf_val, val
end

function discrete_pdf(d::Distribution1D_OLD, u::Int64)::Float64
    u = max(1, u) # JOHN HACK OH GOD WHY
    return d.func[u] / (d.func_int * length(d.func))
end

struct Distribution2D_OLD
    conditional::Vector{Distribution1D_OLD}
    marginal::Distribution1D_OLD

    function Distribution2D_OLD(dat::Matrix)
        # if matrix of floats, proceed, else, convert image to floats
        if dat[1,1] isa Float64
            bw = dat
            nv, nu = size(bw)
        else
            @assert false
            bw = ones(Float64, size(dat))
            nv, nu = size(bw)
            for row = 1:nv
                for col = 1:nu
                    r = convert(Float64, dat[row,col].r)
                    g = convert(Float64, dat[row,col].g)
                    b = convert(Float64, dat[row,col].b)
                    bw[row,col] = max(mean([r,g,b]), .01)
                end
            end
        end

        conditional = Distribution1D_OLD[]
        marginal_func = Float64[]
        for i = 1:nu
            push!(conditional, Distribution1D_OLD(bw[:,i]))
        end
        for i = 1:nu
            push!(marginal_func, conditional[i].func_int)
        end
        return new(
            conditional,
            Distribution1D_OLD(marginal_func)
        )
    end
end

function sample_continuous(d::Distribution2D_OLD, uv::RayTracing.Pnt2)::Tuple{RayTracing.Pnt2, Float64}
    d1, pdf_val1, offset1 = sample_continuous(d.marginal, uv.y)
    d0, pdf_val0, _ = sample_continuous(d.conditional[offset1], uv.x)
    # print("\tpMarginal $(d1)\n")
    # print("\tpConditionalV $(d0)\n")
    @assert (d0 < 1) && (d1 < 1)
    return RayTracing.Pnt2(d0, d1), pdf_val0 * pdf_val1

end

function pdf(d::Distribution2D_OLD, p::RayTracing.Pnt2)::Float64   
    iu = max(1, Int(floor(p.x * length(d.conditional[1].func)))+1) # as opposed to PBRT's clamp
    iv = max(1, Int(floor(p.y * length(d.marginal.func)))+1) # as opposed to PBRT's clamp
    return d.conditional[iv].func[iu] / d.marginal.func_int
end

pdf (generic function with 1 method)

In [50]:
function add_commas(n::Integer)
    s = string(n)
    # Add commas every 3 digits from the right
    return replace(s, r"(?<=[0-9])(?=(?:[0-9]{3})+(?![0-9]))" => ",")
end

add_commas (generic function with 1 method)

In [53]:
for N in 1:14
    data = rand(Float64, 2^N, 2^N)

    d = RayTracing.Distribution2D(data)
    d_old = Distribution2D_OLD(data)

    t = @elapsed xy, pdf_val = RayTracing.sample_continuous(d, RayTracing.Pnt2(0.3, 0.3))

    t_old = @elapsed xy, pdf_val = sample_continuous(d_old, RayTracing.Pnt2(0.3, 0.3))

    println("hehe: $(add_commas(2^N)) x $(add_commas(2^N)) = $(add_commas(2^N * 2^N)) - $(t / t_old)")
end

hehe: 2 x 2 = 4 - 0.16216216216216214
hehe: 4 x 4 = 16 - 0.14285714285714285
hehe: 8 x 8 = 64 - 0.2
hehe: 16 x 16 = 256 - 0.16666666666666666
hehe: 32 x 32 = 1,024 - 0.2857142857142857
hehe: 64 x 64 = 4,096 - 0.2
hehe: 128 x 128 = 16,384 - 0.16666666666666666
hehe: 256 x 256 = 65,536 - 0.06521739130434782
hehe: 512 x 512 = 262,144 - 0.09090909090909091
hehe: 1,024 x 1,024 = 1,048,576 - 0.22115384615384615
hehe: 2,048 x 2,048 = 4,194,304 - 0.13548387096774192
hehe: 4,096 x 4,096 = 16,777,216 - 0.2214765100671141
hehe: 8,192 x 8,192 = 67,108,864 - 0.15999085766527626
hehe: 16,384 x 16,384 = 268,435,456 - 0.156
